<a href="https://colab.research.google.com/github/kshitij730/SHL-RECOMMENDATION-ENGINE/blob/main/Product_Ctalaog_SHL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install Dependencies

In [1]:
!pip install requests beautifulsoup4 pandas lxml

# 2. Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# 3. HTTP Session (critical for Colab stability)

In [3]:
def create_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/120 Safari/537.36"
    })
    return session

session = create_session()


# 4. Test Type Mapping

In [4]:
CATEGORY_MAP = {
    'A': 'Ability & Aptitude',
    'B': 'Biodata & Situational Judgement',
    'C': 'Competencies',
    'D': 'Development & 360',
    'E': 'Assessment Exercises',
    'K': 'Knowledge & Skills',
    'P': 'Personality & Behavior',
    'S': 'Simulations'
}


# 5. Helper Functions

In [5]:
def extract_test_type(td):
    spans = td.find_all("span", class_="product-catalogue__key")
    values = [CATEGORY_MAP.get(s.text.strip(), s.text.strip()) for s in spans]
    return ", ".join(values)

def feature_supported(td):
    return "Yes" if td.find("span", class_="catalogue__circle -yes") else "No"

def extract_description_and_duration(soup):
    description = ""
    duration = ""

    rows = soup.select("div.product-catalogue-training-calendar__row")
    for row in rows:
        h = row.find("h4")
        p = row.find("p")
        if not h or not p:
            continue

        text = h.text.lower()
        if "description" in text:
            description = p.text.strip()
        elif "assessment length" in text:
            duration = p.text.replace(
                "Approximate Completion Time in minutes = ", ""
            ).strip()

    return description, duration


# 6. Core Scraper (Schema-Correct)

In [6]:
def scrape_shl_table(table_type: int, attr: str):
    pages = 32 if table_type == 1 else 12
    base_url = (
        "https://www.shl.com/solutions/products/product-catalog/?start={}&type={}&type={}"
    )

    records = []

    for page in range(pages):
        start = page * 12
        url = base_url.format(start, table_type, table_type)
        print(f"Processing page {page + 1}/{pages}")

        r = session.get(url, timeout=15)
        if r.status_code != 200:
            continue

        soup = BeautifulSoup(r.text, "lxml")
        rows = soup.find_all("tr", attrs={attr: True})

        for row in rows:
            try:
                tds = row.find_all("td")

                assessment_name = row.select_one(
                    "td.custom__table-heading__title"
                ).text.strip()

                assessment_url = "https://www.shl.com" + row.find("a")["href"]

                pr = session.get(assessment_url, timeout=15)
                psoup = BeautifulSoup(pr.text, "lxml")

                description, duration = extract_description_and_duration(psoup)

                records.append({
                    "Assessment Name": assessment_name,
                    "URL": assessment_url,
                    "Description": description,
                    "Assessment Duration": duration,
                    "Test Type": extract_test_type(tds[3]),
                    "Remote Testing Support": feature_supported(tds[1]),
                    "Adaptive/IRT Support": feature_supported(tds[2]),
                })

                print(f"✔ {assessment_name}")
                time.sleep(0.5)

            except Exception as e:
                print(f"✘ Failed row: {e}")

    return pd.DataFrame(records)


# 7. Run + Merge + Save

In [7]:
df1 = scrape_shl_table(1, "data-entity-id")
df2 = scrape_shl_table(2, "data-course-id")

final_df = pd.concat([df1, df2], ignore_index=True)
final_df.sort_values("Assessment Name", inplace=True)

final_df.to_csv("shl_assessments_full_catalog.csv", index=False)

print("Saved shl_assessments_full_catalog.csv")
final_df.head()


Processing page 1/32
✔ Global Skills Development Report
✔ .NET Framework 4.5
✔ .NET MVC (New)
✔ .NET MVVM (New)
✔ .NET WCF (New)
✔ .NET WPF (New)
✔ .NET XAML (New)
✔ Accounts Payable (New)
✔ Accounts Payable Simulation (New)
✔ Accounts Receivable (New)
✔ Accounts Receivable Simulation (New)
✔ ADO.NET (New)
Processing page 2/32
✔ Adobe Experience Manager (New)
✔ Adobe Photoshop CC
✔ Aeronautical Engineering (New)
✔ Aerospace Engineering (New)
✔ Agile Software Development
✔ Agile Testing (New)
✔ AI Skills
✔ Amazon Web Services (AWS) Development (New)
✔ Android Development (New)
✔ Angular 6 (New)
✔ AngularJS (New)
✔ Apache Hadoop (New)
Processing page 3/32
✔ Apache Hadoop Extensions (New)
✔ Apache HBase (New)
✔ Apache Hive (New)
✔ Apache Kafka (New)
✔ Apache Pig (New)
✔ Apache Spark (New)
✔ ASP .NET with C# (New)
✔ ASP.NET 4.5
✔ Assessment and Development Center Exercises
✔ Automata - Fix (New)
✔ Automata - SQL (New)
✔ Automata (New)
Processing page 4/32
✔ Automata Data Science (New)
✔ Au

,Assessment Name,URL,Description,Assessment Duration,Test Type,Remote Testing Support,Adaptive/IRT Support
1,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,The.NET Framework 4.5 test measures knowledge ...,30,Knowledge & Skills,Yes,Yes
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,17,Knowledge & Skills,Yes,No
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,5,Knowledge & Skills,Yes,No
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,11,Knowledge & Skills,Yes,No
5,.NET WPF (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,9,Knowledge & Skills,Yes,No
